# Assignment 2. Sensitivity Analysis: Which Inputs Matter?

**Course:** EPA141A Model Based Decision Making — Delft University of Technology  
**Model:** JUSTICE (development)


## Learning objectives

1. Understand the purpose and limitations of **variance-based sensitivity analysis**.
2. Compute **Sobol first-order (S₁) and total-order (Sₜ)** sensitivity indices using SALib.
3. Apply **Extra-Trees feature importance** as a non-parametric complement.
4. Build a **sensitivity heatmap** comparing input importance across all four objectives.
5. Derive an actionable insight: *which uncertain inputs most deserve monitoring or further research?*


## Background

Sensitivity analysis (SA) asks: **which uncertain inputs most strongly drive outcome variability?**

Answering this helps us decide:
- Where to focus further research (reduce the most influential uncertainties first)
- Which parameters require careful empirical estimation
- Which parameters can be treated as fixed without losing much fidelity

We use two complementary methods:

**Extra-Trees feature importance** fits a random forest of extremely randomised trees
and reports how much each input reduces prediction variance. It is fast, non-parametric,
and captures non-linear effects and interactions — but does not decompose variance formally.

**Sobol variance decomposition** partitions the total output variance into contributions
from each input (S₁) and from all interactions involving that input (Sₜ). It requires
a Saltelli sample (d+2)×N model runs, but provides a rigorous statistical foundation.

### Workflow note
This version uses `feature_scoring.get_feature_scores_all` from the EMA Workbench to
compute Extra-Trees importances without a manual loop. Parameter bounds for the Sobol
`problem` dict are read directly from the EMA model definition, ensuring consistency
between the two methods.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol as sobol_analyze
from ema_workbench import (
    Model, RealParameter, ScalarOutcome,
    perform_experiments, ema_logging,
)
from ema_workbench.em_framework.evaluators import SequentialEvaluator
from ema_workbench.analysis import feature_scoring
from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction
from justice.objectives.objective_functions import years_above_temperature_threshold

os.makedirs("../plots", exist_ok=True)
ema_logging.log_to_stderr(ema_logging.WARNING)

OBJECTIVES = ["welfare", "years_above_temperature_threshold",
              "welfare_loss_damage", "welfare_loss_abatement"]
PARAMS = ["rho", "eta", "delta", "ecs_ensemble"]

years   = np.arange(2015, 2301)
tau_hat = np.clip((years - 2015) / (2100 - 2015), 0, 1)
s_curve = 3 * tau_hat**2 - 2 * tau_hat**3
ECR_REF = np.clip(np.outer(np.full(57, 0.75), s_curve), 0, 1)

print("Imports OK")
print(f"ECR_REF shape: {ECR_REF.shape}")

## Step 1 — Define the model and EMA Workbench wrapper

### Reference policy
A high-abatement Utilitarian policy: ECR S-curve reaching 0.75 at all 57 regions by 2100.

**Task 1.1** — Complete `justice_model_ref` below. It should use `ECR_REF` (the high-abatement reference policy) instead of zero abatement. Wrap it in a `Model` object with four `RealParameter` uncertainties and four `ScalarOutcome` outcomes.

In [ ]:
def justice_model_sa(rho=0.015, eta=1.45, delta=1.0, ecs_ensemble=1, ecr_plateau=0.4):
    """
    EMA Workbench function model interface for JUSTICE.
    Same structure as Assignment 1 — adjust the ECR level via ecr_plateau.
    """
    JUSTICE.hard_reset()
    ensemble_idx = int(np.round(np.clip(ecs_ensemble, 1, 1001)))

    model = JUSTICE(
        start_year=2015, end_year=2300, timestep=1,
        scenario=2,
        climate_ensembles=ensemble_idx,
        stochastic_run=False,
        social_welfare_function=WelfareFunction.UTILITARIAN,
    )

    ecr = np.full((model.no_of_regions, model.no_of_timesteps), ecr_plateau)
    model.run(ecr)
    data = model.evaluate()
    return {
        "welfare":                           data["welfare"][0],
        "years_above_temperature_threshold": data["years_above_temperature_threshold"][0],
        "welfare_loss_damage":               data["welfare_loss_damage"][0],
        "welfare_loss_abatement":            data["welfare_loss_abatement"][0],
    }

# EMA Workbench model object
em_model = Model("JUSTICE", function=justice_model_sa)

em_model.uncertainties = [
    RealParameter("rho",          0.001,  0.030),
    RealParameter("eta",          0.5,    1.5),
    RealParameter("delta",        0.5,    2.0),
    RealParameter("ecs_ensemble", 1,      1001),
]

em_model.levers = [
    RealParameter("ecr_plateau", 0.0, 1.0),
]

em_model.outcomes = [
    ScalarOutcome("welfare"),
    ScalarOutcome("years_above_temperature_threshold"),
    ScalarOutcome("welfare_loss_damage"),
    ScalarOutcome("welfare_loss_abatement"),
]

OBJECTIVES = ["welfare", "years_above_temperature_threshold",
              "welfare_loss_damage", "welfare_loss_abatement"]
PARAMS = ["rho", "eta", "delta", "ecs_ensemble"]

print(em_model)

## Step 2 — LHS ensemble (100 runs)

**Task 2.1** — Run 100 LHS scenarios using `SequentialEvaluator`.
Store the outcomes as `df_results` (a DataFrame) for use with feature scoring.

In [ ]:
POLICY_NAMES = ["no_abatement", "reference_policy"]
policies = [
    Sample("no_abatement",     ecr_plateau=0.0),
    Sample("reference_policy", ecr_plateau=0.4),
]

with SequentialEvaluator(em_model) as evaluator:
    experiments, outcomes = evaluator.perform_experiments(
        scenarios=100, policies=policies)

df_results = pd.DataFrame(outcomes)
df_results["policy"] = experiments["policy"].values
df_params  = experiments[PARAMS]

print(f"Experiments: {experiments.shape}")
print(df_results.groupby("policy")[OBJECTIVES].median().round(3))

## Step 3 — Extra-Trees feature importance

`feature_scoring.get_feature_scores_all(x, y)` fits an `ExtraTreesRegressor` for each
outcome and returns a tidy DataFrame of importances — rows are parameters, columns are outcomes.

**Task 3.1** — Call `feature_scoring.get_feature_scores_all` with:
- `x`: the parameter columns of `experiments`
- `y`: the `outcomes` dict

**Task 3.2** — Plot a 2×2 bar chart grid showing the importance of each parameter per objective.

In [ ]:
# Extra-Trees feature importance scores
# Hint: feature_scoring.get_feature_scores_all(df_params, outcomes)
#       returns a DataFrame of shape (n_params, n_objectives)
scores = ...   # your code here

print(scores.round(3))

# Plot a bar chart of importance per parameter per objective
# (2x2 subplots, one per objective)
colors = ["#4C9BE8", "#E87B4C", "#4CE87B", "#BE4CE8"]
x = np.arange(len(PARAMS))
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, obj in zip(axes.flat, OBJECTIVES):
    # Hint: ax.bar(x, scores[obj], ...)
    ...
    ax.set_xticks(x); ax.set_xticklabels(PARAMS, rotation=20, ha="right")
    ax.set_title(obj); ax.set_ylabel("importance")
fig.suptitle("Extra-Trees feature importance per objective", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", "a02_feature_scores.png"), dpi=120, bbox_inches="tight")
plt.show()

## Step 4 — Sobol sensitivity analysis

Sobol indices require a **Saltelli sample** — a specially designed two-matrix scheme
that allows variance decomposition.

For `d=4` parameters and base sample size `N=32`, SALib generates `N × (D+2) = 192` samples.

The parameter bounds are read directly from `em_model.uncertainties` to ensure they match
the LHS bounds used in Step 2.

**Task 4.1** — Build the SALib `problem` dict from `em_model.uncertainties`, draw Saltelli
samples, run JUSTICE for each, and compute S₁ and Sₜ for all four objectives.

**Task 4.2** — Plot a 2×2 bar chart with S₁ and Sₜ per objective, including confidence-interval error bars.

In [ ]:
# ecs_ensemble is excluded from Morris: it is a discrete index into the FaIR
# ensemble, so local finite-difference derivatives are not well-defined for it.
# Its influence is captured by Extra-Trees in the previous step.

N_MORRIS   = ...   # choose number of trajectories; each costs (n_params+1) model runs
ECS_MEDIAN = ...   # fix ecs_ensemble at a representative value (e.g. median of range)

morris_problem = {
    "num_vars": 3,
    "names":    ["delta", "eta", "rho"],
    "bounds":   [...],   # list of [lower, upper] per parameter (same order as names)
}

X_morris = morris.sample(morris_problem, N_MORRIS, num_levels=4)
print(f"Morris sample size: {X_morris.shape}  (params: {morris_problem['names']})")

morris_results = {}
for pol_name, ecr_val in zip(POLICY_NAMES, [0.0, 0.4]):
    Y = {obj: np.zeros(len(X_morris)) for obj in OBJECTIVES}
    for i, row in enumerate(X_morris):
        delta_v, eta_v, rho_v = row
        out = justice_model_sa(rho=rho_v, eta=eta_v, delta=delta_v,
                               ecs_ensemble=ECS_MEDIAN, ecr_plateau=ecr_val)
        for obj in OBJECTIVES:
            Y[obj][i] = out[obj]
    morris_results[pol_name] = {
        obj: morris.analyze(morris_problem, X_morris, Y[obj]) for obj in OBJECTIVES
    }
    print(f"Morris done: {pol_name}")

In [ ]:
MORRIS_PARAMS = ["delta", "eta", "rho"]
x = np.arange(len(MORRIS_PARAMS))

for pol in POLICY_NAMES:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, obj in zip(axes.flat, OBJECTIVES):
        Si      = morris_results[pol][obj]
        # Hint: Si["mu_star"] is the mean absolute sensitivity, Si["sigma"] the nonlinearity
        # Plot a bar chart of mu_star with error bars for sigma
        ...
        ax.set_xticks(x); ax.set_xticklabels(MORRIS_PARAMS)
        ax.set_title(obj); ax.set_ylabel("mu*")
    fig.suptitle(f"Morris sensitivity — {pol}", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", f"a02_morris_{pol}.png"), dpi=120, bbox_inches="tight")
    plt.show()

## Step 5 — Sensitivity heatmap

**Task 5.1** — Build a heatmap with rows = parameters, columns = objectives,
values = Extra-Trees importance. Normalise each column so values sum to 1.

In [ ]:
# Normalise feature scores per objective so each column sums to 1
imp_norm = scores / scores.sum()

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(imp_norm, annot=True, fmt=".2f", cmap="YlOrRd",
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Sensitivity heatmap — Extra-Trees importance (normalised per objective)")
plt.tight_layout()
plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", "a02_heatmap.png"), dpi=120, bbox_inches="tight")
plt.show()

## Reflection Questions

**1. Why is Morris used instead of Sobol for the normative parameter analysis?**


*(Write your answer here.)*


**2. Physical vs. normative sensitivity.** What do ET and Morris together reveal about
which type of uncertainty dominates each outcome?


*(Write your answer here.)*


**3. Policy-conditional sensitivity.** Which normative parameter's μ\* changed most
between the two policies? What does this imply for decision-making?


*(Write your answer here.)*